# 02 — GRPO Reinforcement Learning

This notebook applies **Group Relative Policy Optimization (GRPO)** on top of
the SFT-tuned model to improve mathematical reasoning.

**Why GRPO?** Unlike SFT which learns from ground-truth labels, GRPO generates
multiple candidate answers per question, scores them with reward functions,
and updates the policy to prefer higher-reward outputs. This lets the model
improve beyond the quality ceiling of the training data.

**Pipeline:**
1. Merge the SFT LoRA adapter into the base model (Unsloth requires a clean model)
2. Load the merged model with Unsloth (2x faster training via kernel fusion)
3. Attach a fresh LoRA adapter for RL training
4. Train with 3 reward functions: boxed format, strict format, correctness

**Dataset:** GSM8K (grade school math — simpler problems with clear numerical answers
provide a cleaner learning signal than competition math)

**Requirements:** Google Colab with A100 GPU (High-RAM recommended).

## 1. Setup

In [1]:
# Clone the repo
!git clone https://github.com/Roogard/math-rl-tuning.git
%cd math-rl-tuning

# Install dependencies
!pip install "protobuf<5" --quiet
!pip install unsloth --quiet
!pip install -e . --quiet
!pip install bitsandbytes latex2sympy2 --quiet

Cloning into 'math-rl-tuning'...
remote: Enumerating objects: 503, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 503 (delta 82), reused 103 (delta 44), pack-reused 352 (from 1)
Receiving objects: 100% (503/503), 4.31 MiB | 14.38 MiB/s, done.
Resolving deltas: 100% (306/306), done.
/content/math-rl-tuning
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 29.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.8 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have p

In [2]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU:     {torch.cuda.get_device_name(0)}")
print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import importlib.util
print(f"Unsloth: {'installed' if importlib.util.find_spec('unsloth') else 'NOT found'}")
print("All imports OK")

PyTorch: 2.10.0+cu128
GPU:     NVIDIA A100-SXM4-80GB
VRAM:    85.1 GB
Unsloth: installed
All imports OK


## 2. Configuration

In [3]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [4]:
from math_rl_tuning.config import load_config
from math_rl_tuning.utils import setup_hf_token, setup_wandb, mount_google_drive

cfg = load_config()

# --- Authentication ---
setup_hf_token()
setup_wandb(cfg.grpo_training.report_to and "math-rl-grpo")

# Mount Google Drive (to load SFT adapter and save RL model)
mount_google_drive()

SFT_ADAPTER_PATH = "/content/drive/MyDrive/math-rl-tuning/sft"

# --- Checkpoint Resume ---
# To resume from a crashed run, paste the checkpoint folder path here.
# Checkpoints are saved every 50 steps to outputs/grpo/checkpoint-*/
# Leave as None to start fresh.
GRPO_CHECKPOINT = None  # e.g. "/content/drive/MyDrive/math-rl-tuning/grpo/checkpoint-50"

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Mounted at /content/drive


## 3. Run GRPO Training

In [ ]:
# Delete old merged model cache (important after SFT config changes).
# merge_adapter() skips the merge if the output directory already exists —
# so if we changed LoRA config or re-ran SFT, we must clear this cache
# to force a fresh merge with the correct weights.
!rm -rf outputs/sft_merged

from math_rl_tuning.grpo_trainer import run_grpo_training

# run_grpo_training:
# 1. Merges SFT LoRA adapter into base model weights (bakes adapter in permanently)
# 2. Loads merged model with Unsloth for faster RL training
# 3. Attaches a NEW LoRA adapter for the RL stage (fresh, untrained weights)
# 4. Loads GSM8K and formats prompts as message dicts for TRL's GRPOTrainer
# 5. Runs GRPO: for each batch, generates num_generations completions per prompt,
#    scores them with 3 reward functions, computes group-relative advantages,
#    and updates the policy with a KL penalty to prevent reward hacking
trainer, model, tokenizer, reward_callback = run_grpo_training(
    cfg,
    sft_adapter_path=SFT_ADAPTER_PATH,
    save_to_drive=True,
    checkpoint_path=GRPO_CHECKPOINT,
)

## 4. Quick Sanity Check

In [ ]:
# Plot reward curves — shows how each reward function evolved over training.
# What to look for:
#   - correctness_reward_func should trend upward (model learning to get answers right)
#   - boxed_format_reward_func should stabilize near the bonus value early
#     (model quickly learns to use \boxed{} format as instructed)
#   - strict_format_reward_func should climb more slowly (harder constraint)
# If correctness stays flat, the model isn't learning — check LR, beta, or dataset.
reward_callback.plot()

In [7]:
from math_rl_tuning.inference import generate

questions = [
    "What is 15% of 240?",
    "Solve for x: 3x + 7 = 22",
    "A rectangle has length 12 cm and width 5 cm. What is its area?",
]

for q in questions:
    print(f"Q: {q}")
    response = generate(q, model, tokenizer)
    print(f"A: {response[:300]}")
    print("-" * 40)

Q: What is 15% of 240?
A: To find 15% of 240, we follow these steps:

Step 1: Convert the percentage to a decimal. 
\[15\% = \frac{15}{100} = 0.15\]

Step 2: Multiply the decimal by the given number.
\[0.15 \times 240 = 36\]

Therefore, 15% of 240 is $\boxed{36}$.
----------------------------------------
Q: Solve for x: 3x + 7 = 22
A: To solve the equation $3x + 7 = 22$, we follow these steps:

1. Subtract 7 from both sides of the equation to isolate the term with the variable:
\[3x + 7 - 7 = 22 - 7\]
This simplifies to:
\[3x = 15\]

2. Divide both sides by 3 to solve for $x$:
\[\frac{3x}{3} = \frac{15}{3}\]
This simplifies to:
\
----------------------------------------
Q: A rectangle has length 12 cm and width 5 cm. What is its area?
A: To find the area of a rectangle, we use the formula:

\[ \text{Area} = \text{Length} \times \text{Width} \]

Given that the length of the rectangle is 12 cm and the width is 5 cm, we substitute these values into the formula:

\[ \text{Area} = 11 \, \te

## 5. Cleanup

In [8]:
from math_rl_tuning.utils import clean_memory

del model, trainer
clean_memory()

Memory cleared.
